In [ ]:


!pip install -q peft accelerate bitsandbytes datasets

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# Load Model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

# LoRA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Dataset
data = [
    {"text": "Q: What is the capital of Ethiopia?\nA: The capital of Ethiopia is Addis Ababa."},
    {"text": "Q: What is the national dish of Ethiopia?\nA: Injera is the national dish made from teff flour."},
    {"text": "Q: Where was coffee discovered?\nA: Coffee was discovered in Ethiopia."},
    {"text": "Q: Tell me about Lalibela.\nA: Lalibela is famous for its rock-hewn churches."},
    {"text": "Q: What is special about Ethiopian culture?\nA: Ethiopia has rich history, diverse ethnic groups, and unique traditions."},
]

dataset = Dataset.from_list(data)

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Training
training_args = TrainingArguments(
    output_dir="ethiopian_domain_adapter",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    fp16=True,
    optim="adamw_8bit",
    report_to="none",
    save_strategy="no"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

# Save
model.save_pretrained("ethiopian_domain_adapter")
tokenizer.save_pretrained("ethiopian_domain_adapter")
print("✅ Fine-tuning completed!")

: 